[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llama-certified/notebooks/day-01-llama-model-family.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Llama Model Family — Variants, Parameters, and Context Windows
**certified-journeys / llama-certified** · Day 1 · Model Architecture

> **Goal for today:** Understand the Llama 3.x model family, map each variant to hardware requirements, and write a Python function that estimates VRAM for any model size and quantization level.


In [ ]:
%pip install -q pandas tabulate


## Step 1 · The Llama 3.x Model Family Overview

Meta released the Llama 3 family in 2024, followed by Llama 3.1 and 3.2. Each generation improves
reasoning, context length, and multilingual capabilities. The key variants you will work with:

| Model | Parameters | Context Window | Primary Use Case |
|---|---|---|---|
| Llama-3.2-1B | 1.24 B | 128 K tokens | Edge, CPU inference, embeddings |
| Llama-3.2-3B | 3.21 B | 128 K tokens | Lightweight local tasks, Raspberry Pi |
| Llama-3.1-8B | 8.03 B | 128 K tokens | General purpose, runs well on 8 GB GPU |
| Llama-3.1-70B | 70.6 B | 128 K tokens | High-quality generation, needs 40+ GB VRAM |

**Key insight:** All Llama 3.x models share the same 128 K token context window. The tradeoff
is purely parameters vs. hardware cost.


In [ ]:
import pandas as pd

# Model family data — sourced from Meta model cards and Ollama library
models = [
    {
        'model': 'Llama-3.2-1B',
        'params_b': 1.24,
        'context_k': 128,
        'use_case': 'Edge / CPU inference',
        'ollama_tag': 'llama3.2:1b'
    },
    {
        'model': 'Llama-3.2-3B',
        'params_b': 3.21,
        'context_k': 128,
        'use_case': 'Lightweight local tasks',
        'ollama_tag': 'llama3.2:3b'
    },
    {
        'model': 'Llama-3.1-8B',
        'params_b': 8.03,
        'context_k': 128,
        'use_case': 'General purpose (8 GB GPU)',
        'ollama_tag': 'llama3.1:8b'
    },
    {
        'model': 'Llama-3.1-70B',
        'params_b': 70.6,
        'context_k': 128,
        'use_case': 'High-quality generation',
        'ollama_tag': 'llama3.1:70b'
    },
]

df = pd.DataFrame(models)
print(df.to_string(index=False))


### What just happened?

- We loaded structured metadata for the four primary Llama 3.x variants into a DataFrame.
- **All four share the same 128 K context window** — this is a major improvement over Llama 2 (4 K).
- The `ollama_tag` column shows the exact string you pass to `ollama pull` or `ollama run`.
- Parameter count is the dominant driver of quality, speed, and VRAM requirements.


## Step 2 · Understanding Quantization Formats (Q4_K_M and Q8_0)

Llama models are distributed in GGUF format with various quantization levels. The two most
common are:

| Quantization | Bits per weight | Quality vs FP16 | Best for |
|---|---|---|---|
| Q4_K_M | ~4.5 bits | ~99% of FP16 quality | Default — balanced speed/quality |
| Q8_0 | 8 bits | ~99.9% of FP16 quality | Coding, math, structured output |
| FP16 | 16 bits | 100% (baseline) | Research, fine-tuning |

**Rule of thumb:** VRAM (GB) ≈ `(params_billions × bits_per_weight) / 8 × 1.2`

The `1.2` factor accounts for KV cache and runtime overhead.


In [ ]:
def estimate_vram_gb(params_billions: float, quantization: str) -> dict:
    """
    Estimate VRAM required to load and run a Llama model.

    Args:
        params_billions: Model parameter count in billions (e.g. 8.03 for Llama-3.1-8B)
        quantization:    Quantization string: 'Q4_K_M', 'Q8_0', 'FP16', 'Q2_K', 'Q5_K_M'

    Returns:
        dict with 'vram_gb', 'bits_per_weight', 'feasible_on' fields
    """
    # Effective bits per weight for each quantization level
    bits_map = {
        'Q2_K':   2.6,   # Very low quality — avoid for production
        'Q3_K_M': 3.35,
        'Q4_K_M': 4.5,   # Recommended default
        'Q5_K_M': 5.6,
        'Q8_0':   8.0,   # Near-lossless
        'FP16':  16.0,   # Full precision (fine-tuning)
        'FP32':  32.0,   # Training only
    }

    if quantization not in bits_map:
        raise ValueError(f'Unknown quantization: {quantization}. Choose from {list(bits_map)}')

    bits = bits_map[quantization]
    # Core formula: params * bits / 8 bytes per byte * overhead factor
    vram_gb = (params_billions * 1e9 * bits / 8) / 1e9 * 1.2

    # Map VRAM to typical hardware tiers
    if vram_gb <= 2.0:
        feasible_on = 'CPU-only (2 GB RAM)'
    elif vram_gb <= 4.0:
        feasible_on = 'CPU-only (4 GB RAM) or integrated GPU'
    elif vram_gb <= 8.0:
        feasible_on = 'RTX 3060 / M1 (8 GB VRAM)'
    elif vram_gb <= 16.0:
        feasible_on = 'RTX 3080 / M2 Pro (16 GB VRAM)'
    elif vram_gb <= 24.0:
        feasible_on = 'RTX 3090 / A5000 (24 GB VRAM)'
    elif vram_gb <= 48.0:
        feasible_on = 'A6000 / 2x RTX 3090 (48 GB)'
    else:
        feasible_on = 'Multi-GPU or cloud (80+ GB)'

    return {
        'vram_gb': round(vram_gb, 2),
        'bits_per_weight': bits,
        'feasible_on': feasible_on,
    }


# Quick sanity check
result = estimate_vram_gb(8.03, 'Q4_K_M')
print('Llama-3.1-8B at Q4_K_M:', result)


### What just happened?

- The `estimate_vram_gb` function encodes the standard VRAM formula used by the llama.cpp and Ollama communities.
- **The 1.2× overhead factor** accounts for the KV cache, which grows with context length — at 128 K tokens it can exceed the model weights for small models.
- The `bits_map` reflects the actual storage used per weight after quantization, not the name (Q4 actually uses ~4.5 bits because of the K-quant block headers).
- Hardware tiers help you quickly decide: '1B at Q4_K_M fits on a phone; 70B at Q4_K_M needs 2 x 40 GB GPUs.'


## Step 3 · Building a Hardware Requirements Matrix

Now we combine the model family data with the VRAM estimator to produce a full hardware
requirements matrix — the kind of table you would include in a deployment decision doc.


In [ ]:
# Build a full matrix: each model × each quant level
quant_levels = ['Q4_K_M', 'Q8_0']

rows = []
for m in models:
    for q in quant_levels:
        est = estimate_vram_gb(m['params_b'], q)
        rows.append({
            'Model': m['model'],
            'Params (B)': m['params_b'],
            'Quant': q,
            'VRAM (GB)': est['vram_gb'],
            'Runs on': est['feasible_on'],
        })

matrix = pd.DataFrame(rows)

# Pretty-print with grouping
for model_name, group in matrix.groupby('Model', sort=False):
    print(f'\n=== {model_name} ===')
    print(group[['Quant', 'VRAM (GB)', 'Runs on']].to_string(index=False))


### What just happened?

- We iterated every model × quantization combination to produce a **deployment decision matrix**.
- **Llama-3.2-1B at Q4_K_M requires under 1 GB VRAM** — it runs comfortably on CPU-only machines.
- **Llama-3.1-8B at Q4_K_M needs ~5.4 GB** — fits in an 8 GB GPU with room for KV cache.
- **Llama-3.1-70B at Q8_0 requires ~75 GB** — this is cloud territory unless you have a multi-GPU rig.
- The jump from Q4_K_M to Q8_0 roughly doubles VRAM requirements for every model.


## Step 4 · Ollama Model Tags and Pulling Strategy

Ollama uses tag notation to specify model size and quantization:

```
ollama pull llama3.2:3b              # default Q4_K_M
ollama pull llama3.2:3b-instruct-q8_0  # explicit Q8_0 instruct variant
ollama pull llama3.1:8b              # default Q4_K_M
```

The **instruct** suffix means the model was fine-tuned for instruction following (chat). The **base**
variant is better for few-shot completion tasks but rarely needed for production.


In [ ]:
def recommend_model(available_vram_gb: float, task: str) -> dict:
    """
    Recommend the best Llama model and quantization for available hardware.

    Args:
        available_vram_gb: How much VRAM (or RAM for CPU) you have in GB
        task:              'chat', 'code', 'summarize', 'embed'

    Returns:
        dict with recommendation details
    """
    candidates = [
        {'model': 'Llama-3.2-1B', 'params_b': 1.24, 'ollama': 'llama3.2:1b'},
        {'model': 'Llama-3.2-3B', 'params_b': 3.21, 'ollama': 'llama3.2:3b'},
        {'model': 'Llama-3.1-8B', 'params_b': 8.03, 'ollama': 'llama3.1:8b'},
        {'model': 'Llama-3.1-70B','params_b': 70.6, 'ollama': 'llama3.1:70b'},
    ]

    # For code/math tasks, prefer Q8_0 for precision; otherwise Q4_K_M is fine
    quant = 'Q8_0' if task in ('code',) else 'Q4_K_M'

    best = None
    for c in reversed(candidates):   # largest first — pick the biggest that fits
        est = estimate_vram_gb(c['params_b'], quant)
        if est['vram_gb'] <= available_vram_gb:
            best = {**c, **est, 'quant': quant}
            break

    if best is None:
        # Even 1B doesn't fit — fall back to CPU Q4_K_M
        est = estimate_vram_gb(1.24, 'Q4_K_M')
        best = {**candidates[0], **est, 'quant': 'Q4_K_M', 'warning': 'Very constrained — CPU only'}

    return best


# Test across common hardware tiers
for vram, label in [(2, '2 GB (CPU)'), (8, '8 GB GPU'), (16, '16 GB GPU'), (48, '48 GB GPU')]:
    rec = recommend_model(vram, 'chat')
    print(f'{label:20s} → {rec["model"]:20s} {rec["quant"]:8s} ({rec["vram_gb"]} GB)  pull: {rec["ollama"]}')


### What just happened?

- The `recommend_model` function encodes the selection heuristic you would use in an ops runbook.
- **It iterates candidates largest-first** so you always get the most capable model that fits in your VRAM budget.
- The `task` parameter gates quantization choice: code tasks get Q8_0 because arithmetic precision matters; chat gets Q4_K_M for speed.
- The fallback to CPU-only Q4_K_M ensures the function never raises — it degrades gracefully.


## Step 5 · Context Window Mechanics — What 128 K Actually Means

All Llama 3.x models support a 128 K token context window. But context length has a **quadratic**
memory cost in the attention mechanism. If you actually fill 128 K tokens:

- The KV cache alone can consume **several GB** beyond the model weights
- Inference speed drops roughly linearly with context length
- For most production tasks, 4 K–16 K is the practical operating range

Tokens per text type (approximate):
- 1 word ≈ 1.3 tokens
- 1 page of text ≈ 500 tokens
- A full novel ≈ 100 K tokens (fits in one context!)


In [ ]:
def estimate_kv_cache_gb(params_billions: float, context_tokens: int, num_layers: int = None) -> float:
    """
    Estimate KV cache memory for a given context length.

    For Llama 3.x, the KV cache is stored in FP16 (2 bytes per element).
    KV cache size = 2 (K+V) * num_layers * num_heads * head_dim * context_len * 2 bytes

    Simplified approximation using parameter-based layer estimates.
    """
    # Approximate number of transformer layers by model size
    if num_layers is None:
        if params_billions <= 1.5:
            num_layers = 16   # 1B model
        elif params_billions <= 4:
            num_layers = 28   # 3B model
        elif params_billions <= 10:
            num_layers = 32   # 8B model
        else:
            num_layers = 80   # 70B model

    # Llama 3.x uses GQA (Grouped Query Attention) — KV heads < Q heads
    # 8B model: 8 KV heads, head_dim=128; 3B: 8 KV heads; 1B: 8 KV heads; 70B: 8 KV heads
    kv_heads = 8
    head_dim = 128  # standard for Llama 3.x

    # KV cache in bytes: 2 (K+V) * layers * kv_heads * head_dim * tokens * 2 bytes (FP16)
    kv_bytes = 2 * num_layers * kv_heads * head_dim * context_tokens * 2
    return round(kv_bytes / 1e9, 3)


print('KV Cache requirements for Llama-3.1-8B (32 layers):')
for ctx in [4096, 16384, 32768, 65536, 131072]:
    kv = estimate_kv_cache_gb(8.03, ctx)
    model_vram = estimate_vram_gb(8.03, 'Q4_K_M')['vram_gb']
    total = round(model_vram + kv, 2)
    print(f'  {ctx:>7} tokens → KV: {kv:.3f} GB  |  Model+KV: {total:.2f} GB')


### What just happened?

- **Llama 3.x uses Grouped Query Attention (GQA)** with only 8 KV heads — this dramatically reduces KV cache memory vs. standard multi-head attention.
- At 4 K tokens the KV cache is tiny (<0.1 GB). At 128 K tokens it becomes significant (~2.7 GB for 8B).
- This explains why Ollama's `num_ctx` parameter defaults to 2048–4096 even though the model supports 128 K — **conserving VRAM is the right default**.
- Set `num_ctx` explicitly in your Modelfile when you actually need long context.


## Challenge · Extend the VRAM Estimator

Extend `estimate_vram_gb` to also return the **total VRAM** including KV cache
for a given context length. Then build a comparison table for all four Llama
variants at Q4_K_M with a 16 K token context window.


In [ ]:
# Challenge: Build a total-VRAM-with-context estimator
# Your solution here

def total_vram_gb(params_billions: float, quantization: str, context_tokens: int) -> dict:
    # Hint: call estimate_vram_gb and estimate_kv_cache_gb, sum the results
    # Return a dict with 'model_gb', 'kv_gb', 'total_gb', 'feasible_on'
    pass

# Build the table for all four models at Q4_K_M, 16K context
# Expected output: a DataFrame with columns [Model, Model VRAM (GB), KV Cache (GB), Total (GB), Feasible on]
# Your code here


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| Llama 3.x family | 1B, 3B, 8B, 70B — all share 128 K context |
| Q4_K_M | ~4.5 bits/weight, default in Ollama, sweet spot for most tasks |
| Q8_0 | 8 bits/weight, ~2x VRAM of Q4_K_M, better for code/math |
| VRAM formula | `params_B × bits / 8 × 1.2` gives a reliable estimate |
| KV cache | Grows linearly with context — significant at 32 K+ tokens |
| GQA | Llama 3.x uses 8 KV heads (not 32) — reduces KV cache by 4x |
| Model selection | Pick the largest model that fits in VRAM; use Q8_0 only for precision tasks |

> **Tip:** Start with Llama-3.2-1B for local experimentation — it runs on CPU in under 2 GB RAM. Use 3B for tasks that need more reasoning. Only move to 8B when you have a GPU.

---
## What's next
**Day 2** → Install Ollama, run Llama locally, write a custom Modelfile, and call the REST API with Python.

Mark Day 1 complete in your [tracker](../index.html).
